# LeetCode 1926 — Nearest Exit from Entrance in Maze bằng Python

Link đề: https://leetcode.com/problems/nearest-exit-from-entrance-in-maze/ · Độ khó: **Medium** · Chủ đề: Array, Breadth-First Search, Matrix

Skeleton LeetCode:

```python
class Solution:
    def nearestExit(self, maze: List[List[str]], entrance: List[int]) -> int:
        ...
```

Lộ trình: **DFS liệt kê mọi đường đơn** (vét cạn) → **DFS có "ghi nhớ khoảng cách tốt nhất"** (thăm lại khi tìm được đường ngắn hơn) → **BFS từ lối vào** `O(m·n)` (lời giải nộp) → **BFS theo tầng, đánh dấu tại chỗ** → **BFS đa nguồn ngược từ mọi lối ra**.

## 1. Mục tiêu học

1. Mô hình hóa mê cung thành **đồ thị lưới không trọng số**: mỗi ô trống là đỉnh, hai ô kề nhau là cạnh.
2. Hiểu vì sao **BFS** cho đường đi **ít bước nhất** trên đồ thị không trọng số, còn DFS thì không.
3. Cài BFS đúng chuẩn: **đánh dấu đã thăm khi đẩy vào hàng đợi** (không phải khi lấy ra), kiểm điều kiện **lối ra** (ô trống trên biên, **khác** lối vào).
4. Biết các biến thể: BFS theo tầng, BFS đa nguồn, BFS có trạng thái mở rộng (phá tường).
5. Kiểm chứng bằng vét cạn trên mê cung nhỏ ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

Cho ma trận `maze` kích thước `m × n` gồm ô trống `'.'` và tường `'+'`, cùng ô xuất phát `entrance = [r, c]`.

Mỗi bước được đi lên/xuống/trái/phải một ô; không được đi vào tường hay ra ngoài mê cung. Một **lối ra** là ô **trống** nằm trên **biên** của mê cung; **lối vào không tính là lối ra**.

Trả về số bước của đường đi ngắn nhất từ lối vào tới lối ra gần nhất, hoặc `-1` nếu không có.

**Ví dụ 1:** `maze = [["+","+",".","+"],[".",".",".","+"],["+","+","+","."]]`, `entrance = [1,2]` → `1` (lối ra `[0,2]` cách 1 bước; `[1,0]` cách 2 bước; `[2,3]` không tới được).

**Ví dụ 2:** `maze = [["+","+","+"],[".",".","."],["+","+","+"]]`, `entrance = [1,0]` → `2` (lối vào `[1,0]` nằm trên biên nhưng không tính).

**Ví dụ 3:** `maze = [[".","+"]]`, `entrance = [0,0]` → `-1`.

**Ràng buộc:** `1 ≤ m, n ≤ 100`, `maze[i][j] ∈ {'.', '+'}`, lối vào luôn là ô trống.

In [1]:
import random
import sys
import time
from collections import deque
from typing import List

def parse(rows):
    return [list(r) for r in rows]

examples = [
    (parse(["++.+", "...+", "+++."]), [1, 2], 1),
    (parse(["+++", "...", "+++"]), [1, 0], 2),
    (parse([".+"]), [0, 0], -1),
]

def show(maze, entrance=None):
    for i, row in enumerate(maze):
        print("  " + " ".join("E" if entrance and [i, j] == list(entrance) else c for j, c in enumerate(row)))

for mz, ent, want in examples:
    show(mz, ent); print("  -> kỳ vọng", want)

  + + . +
  . . E +
  + + + .
  -> kỳ vọng 1
  + + +
  E . .
  + + +
  -> kỳ vọng 2
  E +
  -> kỳ vọng -1


## 3. Kiến thức nền

### 3.1 Bốn hướng và kiểm tra trong lưới

In [2]:
DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
m, n = 3, 4
r, c = 1, 2
print([(r + dr, c + dc) for dr, dc in DIRS if 0 <= r + dr < m and 0 <= c + dc < n])

[(0, 2), (2, 2), (1, 1), (1, 3)]


### 3.2 Ô biên

Ô `(i, j)` nằm trên biên ⇔ `i == 0` hoặc `i == m − 1` hoặc `j == 0` hoặc `j == n − 1`.

In [3]:
def on_border(i, j, m, n):
    return i == 0 or i == m - 1 or j == 0 or j == n - 1

print([on_border(i, j, 3, 4) for i, j in [(0, 2), (1, 1), (2, 3), (1, 0)]])

[True, False, True, True]


### 3.3 `collections.deque` làm hàng đợi FIFO

`append` vào cuối, `popleft` lấy đầu — đều `O(1)`. (`list.pop(0)` là `O(n)` — tránh.)

In [4]:
q = deque([(0, 0)])
q.append((0, 1)); q.append((1, 0))
print(q.popleft(), list(q))

(0, 0) [(0, 1), (1, 0)]


### 3.4 BFS và khoảng cách theo tầng

BFS thăm các đỉnh theo thứ tự khoảng cách tăng dần: tất cả đỉnh cách 1 bước, rồi 2 bước, … Đỉnh đầu tiên thỏa điều kiện mà BFS gặp là đỉnh **gần nhất**.

In [5]:
grid = parse(["...", "...", "..."])
dist = {(1, 1): 0}; q = deque([(1, 1)])
while q:
    i, j = q.popleft()
    for di, dj in DIRS:
        a, b = i + di, j + dj
        if 0 <= a < 3 and 0 <= b < 3 and (a, b) not in dist:
            dist[(a, b)] = dist[(i, j)] + 1; q.append((a, b))
for i in range(3):
    print([dist[(i, j)] for j in range(3)])

[2, 1, 2]
[1, 0, 1]
[2, 1, 2]


### 3.5 Giới hạn đệ quy

DFS đệ quy trên lưới `100 × 100` có thể sâu tới `10⁴` tầng — vượt giới hạn 1000. BFS dùng hàng đợi nên không gặp vấn đề này.

In [6]:
print("recursion limit:", sys.getrecursionlimit())

recursion limit: 1000


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1 (E = lối vào):

```
+ + . +        tầng 0: (1,2) = E
. . E +        tầng 1: (0,2) [biên, trống -> LỐI RA!], (1,1)
+ + + .        tầng 2: (1,0) [lối ra, nhưng xa hơn]
```

BFS loang từ `E` theo từng "vòng sóng". Ngay vòng 1 đã gặp `(0,2)` trên biên ⇒ đáp án 1. Ô `(2,3)` là lối ra nhưng bị tường bao quanh — BFS không bao giờ chạm tới.

Ví dụ 2: `E = (1,0)` nằm trên biên nhưng **không** tính. Vòng 1: `(1,1)` (không phải biên). Vòng 2: `(1,2)` — biên ⇒ đáp án 2.

## 5. Cách 1 — Brute Force / baseline: DFS liệt kê mọi đường đơn

**Ý tưởng:** từ lối vào, DFS mọi đường **không lặp ô**; mỗi khi đứng ở một lối ra, cập nhật độ dài nhỏ nhất. Đúng (đường ngắn nhất luôn là đường đơn) nhưng số đường đơn bùng nổ theo hàm mũ.

In [7]:
def nearest_exit_brute(maze, entrance):
    m, n = len(maze), len(maze[0])
    sr, sc = entrance
    best = [float("inf")]
    seen = {(sr, sc)}
    def dfs(i, j, steps):
        if steps >= best[0]:
            return
        if (i, j) != (sr, sc) and on_border(i, j, m, n):
            best[0] = steps
            return
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if 0 <= a < m and 0 <= b < n and maze[a][b] == "." and (a, b) not in seen:
                seen.add((a, b)); dfs(a, b, steps + 1); seen.remove((a, b))
    dfs(sr, sc, 0)
    return best[0] if best[0] < float("inf") else -1

for mz, ent, want in examples:
    print(nearest_exit_brute(mz, ent), "| kỳ vọng", want)

1 | kỳ vọng 1
2 | kỳ vọng 2
-1 | kỳ vọng -1


**Độ phức tạp:** số đường đơn trên lưới trống có thể là hàm mũ — lưới `6 × 6` trống đã có hàng triệu đường (dù cắt nhánh `steps ≥ best` giúp nhiều). Độ sâu đệ quy tới `m·n`.

**Nút thắt:** một ô được thăm lại qua vô số đường khác nhau; chỉ cần biết **khoảng cách ngắn nhất** tới nó.

In [8]:
open_grid = [["."] * 5 for _ in range(5)]
open_grid[0] = ["+"] * 5; open_grid[4] = ["+"] * 5
for row in open_grid: row[0] = row[4] = "+"
open_grid[0][2] = "."                                  # lối ra duy nhất ở giữa cạnh trên
t = time.perf_counter(); r = nearest_exit_brute(open_grid, [3, 3]); dt = time.perf_counter() - t
show(open_grid, [3, 3]); print(f"brute: {r} trong {1000*dt:.1f} ms")

  + + . + +
  + . . . +
  + . . . +
  + . . E +
  + + + + +
brute: 4 trong 0.0 ms


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** đồ thị **không trọng số** ⇒ BFS từ lối vào cho khoảng cách ngắn nhất tới **mọi** ô; lối ra đầu tiên BFS chạm tới là lối ra **gần nhất**.

**Quy tắc BFS:**
- Đẩy lối vào với khoảng cách 0, đánh dấu đã thăm.
- Lấy ô `(i, j)` ra; *nếu* nó là lối ra (trên biên **và** khác lối vào) *thì* trả khoảng cách.
- Ngược lại, với mỗi ô kề trống **chưa thăm**: đánh dấu **ngay khi đẩy**, khoảng cách `+1`.
- Hàng đợi rỗng ⇒ `-1`.

**Tối ưu nhỏ:** kiểm "lối ra" ngay khi **đẩy** ô kề (thay vì khi lấy ra) — trả sớm hơn một tầng, vẫn đúng vì mọi ô trong cùng tầng có cùng khoảng cách.

**Invariant:** hàng đợi luôn chứa các ô có khoảng cách `d` rồi `d + 1` (không bao giờ lẫn `d + 2`); mỗi ô được đẩy tối đa một lần với khoảng cách đúng.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DFS với "khoảng cách tốt nhất" (thăm lại khi tìm được đường ngắn hơn)

Một bước tiến từ vét cạn: lưu `best[ô]`; DFS chỉ đi tiếp vào ô nếu tới được nó bằng **ít bước hơn** trước. Đúng, nhưng có thể thăm lại một ô nhiều lần (worst-case lớn), và vẫn đệ quy sâu — ta viết bằng stack.

In [9]:
def nearest_exit_dfs_relax(maze, entrance):
    m, n = len(maze), len(maze[0])
    sr, sc = entrance
    best = {(sr, sc): 0}
    stack = [(sr, sc)]
    pushes = 0
    while stack:
        i, j = stack.pop()
        d = best[(i, j)]
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if 0 <= a < m and 0 <= b < n and maze[a][b] == "." and d + 1 < best.get((a, b), float("inf")):
                best[(a, b)] = d + 1; stack.append((a, b)); pushes += 1
    exits = [best[(i, j)] for (i, j) in best if (i, j) != (sr, sc) and on_border(i, j, m, n)]
    return (min(exits) if exits else -1), pushes

for mz, ent, want in examples:
    print(nearest_exit_dfs_relax(mz, ent)[0], want)

1 1
2 2
-1 -1


**Độ phức tạp:** mỗi lần một ô được cải thiện thì bị đẩy lại ⇒ có thể nhiều hơn `m·n` lần đẩy (đo ở mục 11). Nút thắt: DFS đi "sâu" trước nên hay gặp ô bằng đường **dài** trước — BFS thì gặp bằng đường **ngắn nhất** ngay lần đầu.

### 7.2 Cách 3 — BFS từ lối vào (`O(m·n)`) — lời giải chuẩn

In [10]:
def nearest_exit_bfs_demo(maze, entrance):
    m, n = len(maze), len(maze[0])
    sr, sc = entrance
    seen = {(sr, sc)}
    q = deque([(sr, sc, 0)])
    while q:
        i, j, d = q.popleft()
        print(f"lấy ({i},{j}) d={d}")
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if 0 <= a < m and 0 <= b < n and maze[a][b] == "." and (a, b) not in seen:
                if on_border(a, b, m, n):
                    print(f"  ({a},{b}) là lối ra -> trả {d + 1}")
                    return d + 1
                seen.add((a, b)); q.append((a, b, d + 1))
                print(f"  đẩy ({a},{b}) d={d + 1}")
    return -1

print("=>", nearest_exit_bfs_demo(*examples[1][:2]))

lấy (1,0) d=0
  đẩy (1,1) d=1
lấy (1,1) d=1
  (1,2) là lối ra -> trả 2
=> 2


In [11]:
class SolutionBFS:
    def nearestExit(self, maze: List[List[str]], entrance: List[int]) -> int:
        m, n = len(maze), len(maze[0])
        sr, sc = entrance
        seen = [[False] * n for _ in range(m)]
        seen[sr][sc] = True
        q = deque([(sr, sc, 0)])
        while q:
            i, j, d = q.popleft()
            for di, dj in DIRS:
                a, b = i + di, j + dj
                if 0 <= a < m and 0 <= b < n and maze[a][b] == "." and not seen[a][b]:
                    if a == 0 or a == m - 1 or b == 0 or b == n - 1:
                        return d + 1                    # lối vào đã đánh dấu nên không bị tính
                    seen[a][b] = True
                    q.append((a, b, d + 1))
        return -1

for mz, ent, want in examples:
    print(SolutionBFS().nearestExit(mz, ent), want)

1 1
2 2
-1 -1


Vì lối vào được đánh dấu `seen` từ đầu, nó **không bao giờ** được xét như ô kề ⇒ tự động không bị tính là lối ra. **Độ phức tạp:** mỗi ô đẩy tối đa một lần, mỗi ô xét 4 hướng ⇒ `O(m·n)` thời gian và bộ nhớ.

### 7.3 Cách 4 — BFS theo tầng, đánh dấu tại chỗ trên một bản sao

Xử lý cả một tầng mỗi vòng (`for _ in range(len(q))`), khoảng cách là số tầng — không cần lưu `d` trong hàng đợi. Đánh dấu bằng cách biến ô thành `'+'` trên **bản sao** (tránh sửa input của người gọi).

In [12]:
class SolutionLevels:
    def nearestExit(self, maze: List[List[str]], entrance: List[int]) -> int:
        g = [row[:] for row in maze]
        m, n = len(g), len(g[0])
        sr, sc = entrance
        g[sr][sc] = "+"
        q = deque([(sr, sc)])
        steps = 0
        while q:
            steps += 1
            for _ in range(len(q)):
                i, j = q.popleft()
                for di, dj in DIRS:
                    a, b = i + di, j + dj
                    if 0 <= a < m and 0 <= b < n and g[a][b] == ".":
                        if a in (0, m - 1) or b in (0, n - 1):
                            return steps
                        g[a][b] = "+"
                        q.append((a, b))
        return -1

for mz, ent, want in examples:
    print(SolutionLevels().nearestExit(mz, ent), want)

1 1
2 2
-1 -1


### 7.4 Cách 5 — BFS đa nguồn **ngược** từ mọi lối ra

Đẩy **mọi** lối ra (trừ lối vào) vào hàng đợi với khoảng cách 0, BFS cho tới khi chạm lối vào. Khoảng cách tại lối vào = khoảng cách tới lối ra gần nhất (vì đồ thị vô hướng). Hữu ích khi có **nhiều lối vào** cần hỏi (một lần BFS trả lời tất cả — Bài tập 4).

In [13]:
def exit_distances(maze, banned=None):
    m, n = len(maze), len(maze[0])
    dist = [[-1] * n for _ in range(m)]
    q = deque()
    for i in range(m):
        for j in range(n):
            if maze[i][j] == "." and on_border(i, j, m, n) and (i, j) != banned:
                dist[i][j] = 0; q.append((i, j))
    while q:
        i, j = q.popleft()
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if 0 <= a < m and 0 <= b < n and maze[a][b] == "." and dist[a][b] == -1:
                dist[a][b] = dist[i][j] + 1; q.append((a, b))
    return dist

class SolutionMultiSource:
    def nearestExit(self, maze: List[List[str]], entrance: List[int]) -> int:
        sr, sc = entrance
        d = exit_distances(maze, banned=(sr, sc))[sr][sc]
        return d                                  # -1 nếu không tới được; lối vào bị loại khỏi tập nguồn

for mz, ent, want in examples:
    print(SolutionMultiSource().nearestExit(mz, ent), want)

1 1
2 2
-1 -1


**Lưu ý tinh tế:** loại lối vào khỏi tập nguồn là đủ — nếu lối vào nằm trên biên, đường đi từ một lối ra khác **qua** lối vào rồi quay lại không thể ngắn hơn đường không qua nó. **Độ phức tạp:** `O(m·n)`.

## 8. TRACE từng bước

Trace Cách 4 (theo tầng) trên Ví dụ 1:

| tầng `steps` | hàng đợi đầu tầng | ô kề xét | kết luận |
|---|---|---|---|
| 1 | `(1,2)` | `(0,2)` trống, **trên biên** | trả **1** |

Và trên Ví dụ 2:

| tầng | hàng đợi | ô kề | kết luận |
|---|---|---|---|
| 1 | `(1,0)` | `(1,1)` trống, không biên ⇒ đẩy | |
| 2 | `(1,1)` | `(1,2)` trống, trên biên | trả **2** |

In [14]:
def trace_levels(maze, entrance):
    g = [row[:] for row in maze]; m, n = len(g), len(g[0])
    sr, sc = entrance; g[sr][sc] = "+"
    q = deque([(sr, sc)]); steps = 0
    while q:
        steps += 1
        print(f"tầng {steps}: xử lý {list(q)}")
        for _ in range(len(q)):
            i, j = q.popleft()
            for di, dj in DIRS:
                a, b = i + di, j + dj
                if 0 <= a < m and 0 <= b < n and g[a][b] == ".":
                    if a in (0, m - 1) or b in (0, n - 1):
                        print(f"  ({a},{b}) là lối ra -> {steps}"); return steps
                    g[a][b] = "+"; q.append((a, b))
    print("hết hàng đợi -> -1"); return -1

for mz, ent, want in examples:
    trace_levels(mz, ent); print()

tầng 1: xử lý [(1, 2)]
  (0,2) là lối ra -> 1

tầng 1: xử lý [(1, 0)]
tầng 2: xử lý [(1, 1)]
  (1,2) là lối ra -> 2

tầng 1: xử lý [(0, 0)]
hết hàng đợi -> -1



Bản đồ khoảng cách từ mọi lối ra (Cách 5) trên một mê cung lớn hơn (tự dựng) — số ở mỗi ô là số bước tới lối ra gần nhất:

In [15]:
mz = parse(["+++.++", "+....+", "+.++.+", "+....+", "++++.+"])
d = exit_distances(mz)
for i, row in enumerate(mz):
    print("  " + " ".join(f"{d[i][j]:>2}" if c == "." else " +" for j, c in enumerate(row)))

   +  +  +  0  +  +
   +  3  2  1  2  +
   +  4  +  +  2  +
   +  4  3  2  1  +
   +  +  +  +  0  +


## 9. Code nộp LeetCode

In [16]:
from collections import deque
from typing import List

class Solution:
    def nearestExit(self, maze: List[List[str]], entrance: List[int]) -> int:
        m, n = len(maze), len(maze[0])
        sr, sc = entrance
        seen = [[False] * n for _ in range(m)]
        seen[sr][sc] = True                          # lối vào: đã thăm, không bao giờ được xét là lối ra
        q = deque([(sr, sc, 0)])
        while q:
            i, j, d = q.popleft()
            for di, dj in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                a, b = i + di, j + dj
                if 0 <= a < m and 0 <= b < n and maze[a][b] == "." and not seen[a][b]:
                    if a == 0 or a == m - 1 or b == 0 or b == n - 1:
                        return d + 1
                    seen[a][b] = True                # đánh dấu KHI ĐẨY
                    q.append((a, b, d + 1))
        return -1

for mz, ent, want in examples:
    assert Solution().nearestExit(mz, ent) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**BFS cho khoảng cách ngắn nhất.** Bất biến: các ô được đẩy vào hàng đợi theo thứ tự khoảng cách (BFS) không giảm, và khi một ô được đẩy lần đầu với nhãn `d + 1` thì `d + 1` là khoảng cách ngắn nhất tới nó. *Chứng minh quy nạp:* giả sử mọi ô có khoảng cách `≤ d` đã được gán đúng. Ô `v` có khoảng cách thật `d + 1` có hàng xóm `u` với khoảng cách `d`; `u` được lấy ra trước mọi ô khoảng cách `d + 1`, và lúc đó `v` (nếu chưa thăm) được đẩy với nhãn `d + 1`. Không ô nào được gán nhãn nhỏ hơn khoảng cách thật (nhãn là độ dài một đường đi thật).

**Trả đúng lối ra gần nhất.** Lối ra đầu tiên được phát hiện có nhãn `d + 1` nhỏ nhất trong số các lối ra tới được: mọi lối ra gần hơn sẽ đã được phát hiện ở tầng trước. Kiểm lúc **đẩy** vẫn đúng vì trong cùng tầng mọi ô có cùng nhãn.

**Lối vào không bị tính** vì nó được đánh dấu `seen` trước khi BFS bắt đầu — điều kiện `not seen[a][b]` loại nó. Nếu hàng đợi cạn mà chưa gặp lối ra ⇒ không có lối ra nào tới được ⇒ `-1`.

## 11. Phân tích Time / Space

Gọi `V = m·n ≤ 10⁴`.

| Cách | Time | Space |
|---|---|---|
| 1. DFS mọi đường đơn | hàm mũ | `O(V)` |
| 2. DFS + thư giãn | có thể > `O(V)` (thăm lại) | `O(V)` |
| 3. BFS | `O(V)` | `O(V)` |
| 4. BFS theo tầng, tại chỗ | `O(V)` | `O(V)` (bản sao) / `O(hàng đợi)` nếu sửa input |
| 5. BFS đa nguồn ngược | `O(V)` | `O(V)` |

In [17]:
def random_maze(m, n, p_wall, rng):
    return [["+" if rng.random() < p_wall else "." for _ in range(n)] for _ in range(m)]

rng = random.Random(1926)
mz = random_maze(100, 100, 0.25, rng); mz[50][50] = "."
for name, f in [("BFS/Nộp", Solution().nearestExit), ("Theo tầng", SolutionLevels().nearestExit),
                ("Đa nguồn", SolutionMultiSource().nearestExit)]:
    t = time.perf_counter(); r = f(mz, [50, 50])
    print(f"{name}: {r} trong {1000*(time.perf_counter()-t):.1f} ms")
r2, pushes = nearest_exit_dfs_relax(mz, [50, 50])
print(f"DFS + thư giãn: {r2} với {pushes} lần đẩy (số ô trống ≈ {sum(row.count('.') for row in mz)})")

BFS/Nộp: 59 trong 2.3 ms
Theo tầng: 59 trong 1.9 ms
Đa nguồn: 59 trong 4.0 ms


DFS + thư giãn: 59 với 3472748 lần đẩy (số ô trống ≈ 7584)


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng, đối chiếu vét cạn)

In [18]:
sols = [Solution(), SolutionBFS(), SolutionLevels(), SolutionMultiSource()]
edge = examples + [
    (parse(["."]), [0, 0], -1),                  # 1x1: lối vào là ô biên duy nhất
    (parse([".."]), [0, 0], 1),
    (parse(["...", "...", "..."]), [1, 1], 1),
    (parse(["+.+", "+.+", "+.+"]), [1, 1], 1),
    (parse(["+++", "+.+", "+++"]), [1, 1], -1),  # bị vây kín
    (parse(["+.+++", "+...+", "+++.+", "+...+", "+.+++"]), [2, 3], 4),
]
for mz, ent, want in edge:
    assert nearest_exit_brute(mz, ent) == want, (mz, ent, nearest_exit_brute(mz, ent))
    for s in sols:
        assert s.nearestExit(mz, ent) == want, (mz, ent, s)
    print("OK", ["".join(r) for r in mz], ent, "->", want)

OK ['++.+', '...+', '+++.'] [1, 2] -> 1
OK ['+++', '...', '+++'] [1, 0] -> 2
OK ['.+'] [0, 0] -> -1
OK ['.'] [0, 0] -> -1
OK ['..'] [0, 0] -> 1
OK ['...', '...', '...'] [1, 1] -> 1
OK ['+.+', '+.+', '+.+'] [1, 1] -> 1
OK ['+++', '+.+', '+++'] [1, 1] -> -1
OK ['+.+++', '+...+', '+++.+', '+...+', '+.+++'] [2, 3] -> 4


### 12.2 Mê cung ngẫu nhiên seed cố định: so với vét cạn

In [19]:
rng = random.Random(42)
cnt = 0
for _ in range(600):
    m, n = rng.randint(1, 5), rng.randint(1, 5)
    mz = random_maze(m, n, rng.choice([0.2, 0.35, 0.5]), rng)
    empties = [(i, j) for i in range(m) for j in range(n) if mz[i][j] == "."]
    if not empties:
        continue
    ent = list(rng.choice(empties))
    want = nearest_exit_brute(mz, ent)
    for s in sols:
        assert s.nearestExit(mz, ent) == want, (mz, ent)
    assert nearest_exit_dfs_relax(mz, ent)[0] == want
    cnt += 1
for _ in range(30):
    mz = random_maze(60, 60, 0.3, rng); mz[30][30] = "."
    assert len({s.nearestExit(mz, [30, 30]) for s in sols}) == 1
print(f"{cnt} mê cung nhỏ (vét cạn) + 30 mê cung 60x60: khớp")

573 mê cung nhỏ (vét cạn) + 30 mê cung 60x60: khớp


### 12.3 Không sửa input

In [20]:
mz = parse(["+.+", "...", "+.+"]); before = [row[:] for row in mz]
for s in sols:
    s.nearestExit(mz, [1, 1])
assert mz == before
print("các lời giải không làm thay đổi maze của người gọi")

các lời giải không làm thay đổi maze của người gọi


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Tính lối vào là lối ra

In [21]:
def wrong_entrance_exit(maze, entrance):
    m, n = len(maze), len(maze[0]); sr, sc = entrance
    seen = {(sr, sc)}; q = deque([(sr, sc, 0)])
    while q:
        i, j, d = q.popleft()
        if on_border(i, j, m, n):                       # SAI: lối vào cũng thỏa
            return d
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if 0 <= a < m and 0 <= b < n and maze[a][b] == "." and (a, b) not in seen:
                seen.add((a, b)); q.append((a, b, d + 1))
    return -1

mz, ent, want = examples[1]
print("sai:", wrong_entrance_exit(mz, ent), "| đúng:", want)

sai: 0 | đúng: 2


**Vì sao sai:** đề nói rõ "lối vào không tính là lối ra". Lối vào `[1,0]` nằm trên biên ⇒ bản sai trả 0. **Sửa:** chỉ xét các ô **khác** lối vào (đánh dấu lối vào đã thăm và kiểm khi đẩy ô kề, hoặc thêm điều kiện `(i, j) != (sr, sc)`).

### Lỗi 2 — Dùng DFS và trả đường đầu tiên tìm thấy

In [22]:
def wrong_first_dfs(maze, entrance):
    m, n = len(maze), len(maze[0]); sr, sc = entrance
    seen = {(sr, sc)}
    stack = [(sr, sc, 0)]
    while stack:
        i, j, d = stack.pop()
        if (i, j) != (sr, sc) and on_border(i, j, m, n):
            return d                                    # SAI: lối ra đầu tiên DFS gặp chưa chắc gần nhất
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if 0 <= a < m and 0 <= b < n and maze[a][b] == "." and (a, b) not in seen:
                seen.add((a, b)); stack.append((a, b, d + 1))
    return -1

mz = parse(["+.+++", "+...+", "+.+.+", "+...+", "+++.+"])
print("DFS:", wrong_first_dfs(mz, [2, 1]), "| BFS:", Solution().nearestExit(mz, [2, 1]))

DFS: 4 | BFS: 2


**Vì sao sai:** DFS đi sâu theo một hướng trước, có thể tới một lối ra xa trước khi thử hướng gần. Lối ra đầu tiên DFS gặp **không** bảo đảm gần nhất. **Sửa:** BFS.

### Lỗi 3 — Đánh dấu đã thăm khi **lấy ra** thay vì khi **đẩy vào**

In [23]:
def mark_on_pop(maze, entrance):
    m, n = len(maze), len(maze[0]); sr, sc = entrance
    seen = set(); q = deque([(sr, sc, 0)]); pushes = 0
    while q:
        i, j, d = q.popleft()
        if (i, j) in seen:
            continue
        seen.add((i, j))                                # đánh dấu muộn
        if (i, j) != (sr, sc) and on_border(i, j, m, n):
            return d, pushes
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if 0 <= a < m and 0 <= b < n and maze[a][b] == "." and (a, b) not in seen:
                q.append((a, b, d + 1)); pushes += 1
    return -1, pushes

big = [["."] * 60 for _ in range(60)]
for i in range(60): big[i][0] = big[i][59] = "+"
for j in range(60): big[0][j] = big[59][j] = "+"
big[0][30] = "."                                         # lối ra duy nhất
r, pushes = mark_on_pop(big, [58, 30])
print(f"đánh dấu khi lấy ra: kết quả {r}, số lần đẩy {pushes} (số ô trống ≈ {58*58 + 1})")
print("đánh dấu khi đẩy:", Solution().nearestExit(big, [58, 30]))

đánh dấu khi lấy ra: kết quả 58, số lần đẩy 5045 (số ô trống ≈ 3365)
đánh dấu khi đẩy: 58


**Vì sao chưa tốt:** kết quả vẫn đúng, nhưng một ô có thể bị đẩy **nhiều lần** (từ nhiều hàng xóm cùng tầng) ⇒ hàng đợi phình to gấp mấy lần số ô. Nếu còn **quên** kiểm `seen` khi lấy ra thì sẽ lặp vô hạn. **Sửa:** đánh dấu ngay khi đẩy.

### Lỗi 4 — Kiểm biên sai chỉ số (`i == m` thay vì `i == m − 1`)

In [24]:
def wrong_border(maze, entrance):
    m, n = len(maze), len(maze[0]); sr, sc = entrance
    seen = {(sr, sc)}; q = deque([(sr, sc, 0)])
    while q:
        i, j, d = q.popleft()
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if 0 <= a < m and 0 <= b < n and maze[a][b] == "." and (a, b) not in seen:
                if a == 0 or a == m or b == 0 or b == n:          # SAI: m, n nằm ngoài lưới
                    return d + 1
                seen.add((a, b)); q.append((a, b, d + 1))
    return -1

for mz, ent, want in examples:
    print("sai:", wrong_border(mz, ent), "| đúng:", want)

sai: 1 | đúng: 1
sai: -1 | đúng: 2
sai: -1 | đúng: -1


**Vì sao sai:** chỉ số hợp lệ là `0..m−1`; hàng cuối là `m − 1`. Kiểm `== m` không bao giờ đúng ⇒ bỏ sót mọi lối ra ở cạnh dưới/phải (Ví dụ 2 ra `-1`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Số bước ít nhất trên lưới/đồ thị không trọng số"** ⇒ BFS, không DFS.
- Đánh dấu thăm **khi đẩy**; điều kiện đích có thể kiểm khi đẩy để trả sớm.
- **Nhiều nguồn / nhiều đích** ⇒ BFS đa nguồn (đẩy hết nguồn vào hàng đợi ban đầu) — LeetCode 542 *01 Matrix*, 994 *Rotting Oranges*, 1162.
- Trạng thái phức tạp hơn vị trí (phá tường, chìa khóa) ⇒ BFS trên **(vị trí, trạng thái)** — LeetCode 1293, 864.
- Cẩn thận lối vào trên biên.

## 15. Biến thể của bài

1. **Nhiều lối vào, mỗi lối vào hỏi khoảng cách tới lối ra gần nhất** ⇒ một lần BFS đa nguồn từ các lối ra (Bài tập 4).
2. **Được phá tối đa `k` bức tường** ⇒ BFS trên trạng thái `(i, j, số tường đã phá)` — LeetCode 1293 (Bài tập 5 với `k = 1`).
3. **Ô có chi phí khác nhau** ⇒ Dijkstra; chi phí 0/1 ⇒ 0-1 BFS.
4. **Trả về chính đường đi** ⇒ lưu `parent` trong BFS (Bài tập 3).
5. **Lưới rất lớn / mê cung thưa** ⇒ BFS hai chiều (từ lối vào và từ các lối ra).

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. DFS mọi đường | vét cạn | mũ | `O(V)` | kiểm chứng |
| 2. DFS + thư giãn | cập nhật khi ngắn hơn | > `O(V)` | `O(V)` | để thấy vì sao cần BFS |
| 3. BFS | sóng loang từ lối vào | `O(V)` | `O(V)` | **nộp bài** |
| 4. BFS theo tầng | đếm tầng, đánh dấu tại chỗ | `O(V)` | `O(V)` | không cần lưu `d` |
| 5. BFS đa nguồn | từ mọi lối ra | `O(V)` | `O(V)` | nhiều truy vấn lối vào |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `maze = ["..",".."]`, lối vào `[0,0]`; và `maze = ["+.+","+.+"]`, lối vào `[1,1]`?

**Bài 2 (tính tay):** Ghi các tầng BFS cho mê cung `["+.+++","+...+","+++.+","+...+","+.+++"]`, lối vào `[2,3]`.

**Bài 3 (điền TODO):** Viết `nearest_exit_path(maze, entrance)` trả về **danh sách ô** trên một đường ngắn nhất (từ lối vào tới lối ra), hoặc `[]`.

**Bài 4 (tự viết lại):** Viết `answer_many(maze, entrances)` trả lời nhiều lối vào bằng **một** lần BFS đa nguồn. (Chú ý: đề yêu cầu lối vào không được là lối ra của chính nó.)

**Bài 5 (mini challenge):** Được **phá tối đa một** bức tường. Viết `nearest_exit_break_one(maze, entrance)` bằng BFS trên trạng thái `(i, j, đã_phá)`.

In [25]:
def nearest_exit_path(maze, entrance):
    # TODO
    pass

def answer_many(maze, entrances):
    # TODO
    pass

def nearest_exit_break_one(maze, entrance):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Lưới 2x2 mọi ô đều biên | `[0,1]` cách 1 bước | `["+.+","+.+"]`: `(0,1)` cách 1 |
| 2 | Tầng 1: `(1,3)`, `(3,3)` | Tầng 2: `(1,2)`, `(3,2)`; tầng 3: `(1,1)`, `(3,1)` | Tầng 4: `(0,1)` hoặc `(4,1)` là lối ra ⇒ 4 |
| 3 | Lưu `parent[ô]` khi đẩy | Gặp lối ra thì lần ngược | Đảo danh sách |
| 4 | BFS đa nguồn cho khoảng cách mọi ô | Lối vào trên biên: phải loại chính nó khỏi nguồn | Với lối vào trên biên chạy BFS riêng (hoặc lấy khoảng cách tới lối ra **khác**) |
| 5 | Trạng thái `(i, j, b)` với `b ∈ {0, 1}` | Đi vào tường chỉ khi `b = 0`, chuyển sang `b = 1` | Vét cạn: thử phá từng bức tường rồi BFS |

## 19. Đáp án tham khảo

In [26]:
r1 = [Solution().nearestExit(parse(["..", ".."]), [0, 0]), Solution().nearestExit(parse(["+.+", "+.+"]), [1, 1])]
print("Bài 1:", r1); assert r1 == [1, 1]
print("Bài 2:"); assert trace_levels(parse(["+.+++", "+...+", "+++.+", "+...+", "+.+++"]), [2, 3]) == 4

Bài 1: [1, 1]
Bài 2:
tầng 1: xử lý [(2, 3)]
tầng 2: xử lý [(1, 3), (3, 3)]
tầng 3: xử lý [(1, 2), (3, 2)]
tầng 4: xử lý [(1, 1), (3, 1)]
  (0,1) là lối ra -> 4


In [27]:
def nearest_exit_path(maze, entrance):
    m, n = len(maze), len(maze[0]); s = tuple(entrance)
    parent = {s: None}; q = deque([s])
    while q:
        i, j = q.popleft()
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if 0 <= a < m and 0 <= b < n and maze[a][b] == "." and (a, b) not in parent:
                parent[(a, b)] = (i, j)
                if on_border(a, b, m, n):
                    path, cur = [], (a, b)
                    while cur is not None:
                        path.append(list(cur)); cur = parent[cur]
                    return path[::-1]
                q.append((a, b))
    return []

def answer_many(maze, entrances):
    m, n = len(maze), len(maze[0])
    base = exit_distances(maze)
    out = []
    for r, c in entrances:
        if on_border(r, c, m, n):
            out.append(Solution().nearestExit(maze, [r, c]))      # lối vào trên biên: loại chính nó
        else:
            out.append(base[r][c])
    return out

rng = random.Random(3)
for _ in range(300):
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    mz = random_maze(m, n, 0.3, rng)
    empties = [[i, j] for i in range(m) for j in range(n) if mz[i][j] == "."]
    if not empties: continue
    ent = rng.choice(empties)
    p = nearest_exit_path(mz, ent)
    want = Solution().nearestExit(mz, ent)
    assert (len(p) - 1 if p else -1) == want
    if p:
        assert p[0] == ent and on_border(*p[-1], m, n) and all(mz[i][j] == "." for i, j in p)
        assert all(abs(p[t][0] - p[t + 1][0]) + abs(p[t][1] - p[t + 1][1]) == 1 for t in range(len(p) - 1))
    qs = rng.sample(empties, min(3, len(empties)))
    assert answer_many(mz, qs) == [Solution().nearestExit(mz, q) for q in qs]
print("Bài 3, 4: OK", nearest_exit_path(*examples[0][:2]))

Bài 3, 4: OK [[1, 2], [0, 2]]


In [28]:
def nearest_exit_break_one(maze, entrance):
    m, n = len(maze), len(maze[0]); sr, sc = entrance
    seen = {(sr, sc, 0)}; q = deque([(sr, sc, 0, 0)])
    while q:
        i, j, used, d = q.popleft()
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if not (0 <= a < m and 0 <= b < n):
                continue
            nu = used + (maze[a][b] == "+")
            if nu > 1 or (a, b, nu) in seen:
                continue
            if on_border(a, b, m, n) and (a, b) != (sr, sc):
                return d + 1                  # ô biên (trống, hoặc tường vừa phá) là lối ra
            seen.add((a, b, nu)); q.append((a, b, nu, d + 1))
    return -1

def break_one_brute(maze, entrance):
    best = Solution().nearestExit(maze, entrance)
    for i in range(len(maze)):
        for j in range(len(maze[0])):
            if maze[i][j] == "+":
                g = [row[:] for row in maze]; g[i][j] = "."
                r = Solution().nearestExit(g, entrance)
                if r != -1 and (best == -1 or r < best):
                    best = r
    return best

rng = random.Random(8)
for _ in range(400):
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    mz = random_maze(m, n, 0.45, rng)
    empties = [[i, j] for i in range(m) for j in range(n) if mz[i][j] == "."]
    if not empties: continue
    ent = rng.choice(empties)
    assert nearest_exit_break_one(mz, ent) == break_one_brute(mz, ent), (mz, ent)
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK",
      nearest_exit_break_one(parse(["+++", "+.+", "+++"]), [1, 1]))

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK 1


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
seen = [[False] * n for _ in range(m)]; seen[sr][sc] = True
q = deque([(sr, sc, 0)])
while q:
    i, j, d = q.popleft()
    for di, dj in ((-1,0), (1,0), (0,-1), (0,1)):
        a, b = i + di, j + dj
        if 0 <= a < m and 0 <= b < n and maze[a][b] == "." and not seen[a][b]:
            if a in (0, m - 1) or b in (0, n - 1):
                return d + 1
            seen[a][b] = True
            q.append((a, b, d + 1))
return -1
```

**Checklist:**
- [ ] BFS (không DFS) vì cần **ít bước nhất**?
- [ ] Lối vào **không** tính là lối ra (đánh dấu trước, kiểm khi đẩy ô kề)?
- [ ] Đánh dấu thăm **khi đẩy**?
- [ ] Biên là `0` và `m − 1` / `n − 1`?
- [ ] Không có lối ra tới được ⇒ `-1`; không sửa input (hoặc biết mình đang sửa).